# Training data download
Runs `Code/download_training.py`: new buildings first, then buildings last downloaded over 60 days ago, at most 1,000 RealtyAPI requests per run.

Needs two Colab secrets (key icon in the left bar): `REALTYAPI_KEY`, and `GITHUBTOKEN` for the last cell, which pushes the new files to GitHub.

In [ ]:
import os
os.chdir("/content")
if os.path.exists("structure_properties_fall_2026"):
    !git -C structure_properties_fall_2026 pull origin main
else:
    !git clone https://github.com/aldenbarnes/structure_properties_fall_2026.git
os.chdir("/content/structure_properties_fall_2026")

Optional: see what a run would request (uses only the search requests).

In [ ]:
from google.colab import userdata
os.environ["REALTYAPI_KEY"] = userdata.get("REALTYAPI_KEY")
!python Code/download_training.py --dry-run

Download. Lower the cap with `--max-requests 200` if you want a smaller run.

In [ ]:
!python Code/download_training.py

Save the new files to GitHub.

In [ ]:
import subprocess, datetime
token = userdata.get("GITHUBTOKEN")
url = "https://x-access-token:{}@github.com/aldenbarnes/structure_properties_fall_2026.git".format(token)
day = datetime.date.today().isoformat()
steps = [["git", "add", "Data/training", "Data/realty_training_combined.csv"],
         ["git", "-c", "user.name=Training download", "-c", "user.email=training-download@users.noreply.github.com",
          "commit", "-m", "Training data download " + day],
         ["git", "pull", "--no-rebase", url, "main"],
         ["git", "push", url, "HEAD:main"]]
for cmd in steps:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    if r.returncode:
        break